# Notebook 03 — Does Our Search Find the Right Records?

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

In notebook 02 we stored 337 records in Pinecone. Now we check: when we ask a question, does the search bring back
the **right** records? We use an **answer key** — questions where we already know the right answer — and count.

```text
answer key row                       vector DB                          compare
question + right ids       ──►   search(question) → top ids     ──►   which of them are right?
                                                                            │
                                                                            ▼
                                                            metrics for THIS question
                                                                            │
                                                                            ▼
                                                  one row per question → averages per sheet → MLflow
```

```text
0 SETUP           connect, open the answer key, the search function
1 ONE QUESTION    search it, look at the top 3
2 THE METRICS     five metrics, by hand, and ONE function that calculates them
3 ALL QUESTIONS   the same for every question → one row per question
4 AVERAGES        one line per sheet + the questions that failed
5 SAVE            MLflow
```

**You need:** notebook 02 run once, your `.env` file, and `ground_truth_ecom.xlsx` in the `docs` folder.

## 0 — Setup

Run these cells in order. Each one prints a ✅ when it worked.

In [7]:
# Install the libraries (only needed once on your computer).
%pip install -q openai pinecone python-dotenv openpyxl mlflow


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**0.1 — Connect** to OpenAI and Pinecone, with the same model, index and namespace as notebook 02.

In [8]:
import csv
import math
import os
from datetime import datetime

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

# STEP 1 — read the API keys from the .env file
load_dotenv()

# STEP 2 — stop early with a clear message if a key is missing
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

# STEP 3 — the same settings as notebook 02
EMBED_MODEL = "text-embedding-3-small"     # MUST be the model notebook 02 used to store the records
INDEX_NAME = "ecom-kb-tutorial"
NAMESPACE = "ecom-kb"

# STEP 4 — how many results we look at
K_ONE = 3        # questions with ONE right answer: the agent's lookups return the top 3
K_MANY = 5       # golden questions with SEVERAL right answers: a little more room

# STEP 5 — connect
client = OpenAI()
index = Pinecone(api_key=os.getenv("PINECONE_API_KEY")).Index(INDEX_NAME)

print("✅ Connected to", INDEX_NAME)

✅ Connected to ecom-kb-tutorial


**0.2 — Open the answer key.** Two small helpers: `read_sheet()` turns a sheet into a list of rows, and
`split_ids()` turns a cell holding several ids (one per line) into a Python list.

In [9]:
# Open the Excel file in read-only mode (we only read it).
answer_key = openpyxl.load_workbook("./docs/ground_truth_ecom.xlsx", read_only=True)


def read_sheet(sheet_name):
    """Read one sheet and return its rows as a list of dictionaries.

    Example of one row: {"qid": "T03", "question": "How much does a typical checkout bring in?",
                         "right_id": "glossary::Average Order Value", ...}
    """
    rows = answer_key[sheet_name].iter_rows(values_only=True)

    # The first row holds the column names.
    column_names = next(rows)

    records = []
    for values in rows:
        # Skip empty rows.
        if all(value is None for value in values):
            continue

        # Pair each column name with its value in this row.
        record = {}
        for column_name, value in zip(column_names, values):
            record[column_name] = value
        records.append(record)

    return records


def split_ids(cell):
    """Turn a cell with one id per line into a list.

    "glossary::Revenue
     column::ecom.orders.total_amount"   ->   ["glossary::Revenue", "column::ecom.orders.total_amount"]
    """
    if not cell:
        return []

    ids = []
    for line in str(cell).split("\n"):
        line = line.strip()
        if line:
            ids.append(line)
    return ids

In [10]:
# Read the four sheets we test, each into its own list.
glossary_questions = read_sheet("glossary_questions")    # one business word, said in other words  → 1 right id
example_questions = read_sheet("example_questions")      # one saved SQL example, asked differently → 1 right id
join_questions = read_sheet("join_questions")            # one join between two tables             → 1 right id
golden_questions = read_sheet("golden")                  # real business questions                 → several

print("✅ Answer key opened")
print("   glossary questions:", len(glossary_questions))
print("   example questions: ", len(example_questions))
print("   join questions:    ", len(join_questions))
print("   golden questions:  ", len(golden_questions))

✅ Answer key opened
   glossary questions: 66
   example questions:  61
   join questions:     16
   golden questions:   49


**0.3 — The search.** `search()` turns the question into numbers (an embedding) and asks Pinecone for the closest
records. `tabs` limits the search to some tabs of the knowledge base — the same filters the agent uses.

In [11]:
def search(question, tabs, top_k):
    """
    Search Pinecone for records most similar to a question.

    Args:
        question: The text to search for.
        tabs: Optional list of sheet names to search within. Use None or an
            empty list to search across the full namespace.
        top_k: Maximum number of results to return.

    Returns:
        A list of record IDs, ordered from closest match to less close match.

    This function embeds the question using the same embedding model used for
    the stored records, then asks Pinecone to find the nearest vectors.
    """

    # STEP 1 — Convert the question into an embedding vector.
    # An embedding is a list of numbers that represents the meaning of the
    # question. It must use the same model as the vectors already stored in
    # Pinecone so both sides can be compared meaningfully.
    #
    # The API accepts a list of input strings, so the single question is
    # wrapped in a one-item list. The response's first data item corresponds
    # to that question.
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=[question],
    )
    question_vector = response.data[0].embedding

    # STEP 2 — Build an optional metadata filter.
    # Start with None, which means no filter is applied and all records in the
    # selected namespace are eligible for search.
    search_filter = None

    # If tabs were provided, restrict the search to records whose "sheet"
    # metadata value is one of the requested tab names.
    # Pinecone's "$in" condition matches any value in the supplied list.
    # An empty tabs list is treated like no filter because it is falsy.
    if tabs:
        search_filter = {"sheet": {"$in": tabs}}

    # STEP 3 — Query Pinecone for the nearest stored vectors.
    # "vector" is the question embedding from Step 1.
    # "top_k" sets the maximum number of matches to return.
    # "namespace" limits the search to the configured namespace.
    # "include_metadata=True" asks Pinecone to include each match's metadata
    # in the response, although this function only returns IDs.
    # "filter" applies the optional sheet restriction from Step 2.
    results = index.query(
        vector=question_vector,
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=True,
        filter=search_filter,
    )

    # STEP 4 — Extract just the record IDs.
    # Pinecone returns matches ordered from highest similarity to lowest
    # similarity, so appending each ID in response order preserves the ranking:
    # retrieved_ids[0] is the closest match.
    retrieved_ids = []
    for match in results.matches:
        retrieved_ids.append(match.id)

    # Return only IDs; callers that need scores or metadata would need to
    # return those fields separately.
    return retrieved_ids


# Try the search once with no tab filter, so all records in NAMESPACE can
# compete. Request at most five results for this example question.
print("✅ Search works. Top 5 for 'What was our GMV last month?':")
for record_id in search(
    "What was our GMV last month?",
    tabs=None,
    top_k=5,
):
    # Print each ID on its own indented line, in Pinecone's ranked order.
    print("   ", record_id)

✅ Search works. Top 5 for 'What was our GMV last month?':
    nlq_examples::8
    nlq_examples::1
    glossary::Revenue
    nlq_examples::43
    nlq_examples::23


## 1 — One question

Take one row of the answer key: the question, and the record that **should** come back.

**Guess first:** *"How much does a typical checkout bring in?"* — the right answer is
`glossary::Average Order Value`. The question doesn't use those words. Will the search find it in the top 3?

In [12]:
# Choose one question from glossary_questions.
# Change the index from 0 to 65 to inspect a different question.
row = glossary_questions[2]

# Read the question text from the selected row.
question = row["question"]

# Convert the answer key's right_id value into a list of expected record IDs.
# split_ids handles the format used in the source data.
expected_ids = split_ids(row["right_id"])


# Search only the glossary and capability_card tabs, where the correct
# records for this question are expected to live.
# K_ONE controls how many top-ranked results to request.
retrieved_ids = search(
    question,
    tabs=["glossary", "capability_card"],
    top_k=K_ONE,
)


# Print the question and its expected answer IDs so the search results below
# can be compared with the answer key.
print("Question: ", question)
print("Expected: ", expected_ids)
print()


# Display each retrieved ID with its rank.
# The first result is rank 1, the second is rank 2, and so on.
# Mark an ID as "right" when it appears in the expected answer list.
place = 1
for record_id in retrieved_ids:
    if record_id in expected_ids:
        print(f"  {place}. {record_id}   ✅ right")
    else:
        print(f"  {place}. {record_id}")

    # Advance the rank for the next result.
    place += 1

Question:  How much does a typical checkout bring in?
Expected:  ['glossary::Average Order Value']

  1. glossary::Revenue
  2. capability_card::3
  3. glossary::Percentage Coupon


That is the whole idea: **search, then compare the ids with the answer key.** Next we turn "how good was this?" into
numbers.

## 2 — The metrics, by hand

For every question we compare two lists:

- **retrieved ids** — what the search returned, best first
- **expected ids** — what the answer key says is right

Take the GMV question. The answer key expects **2** records: `glossary::Revenue` and
`column::ecom.orders.total_amount`. Say the search returned 5 ids, and only Revenue (in 1st place) is right:

```text
place:      1      2      3      4      5
right?      ✓      ✗      ✗      ✗      ✗
```

| Metric | Question it answers | Formula | This example |
|---|---|---|---:|
| **Hit** | did we find **anything** right? | 1 if at least one ✓, else 0 | 1 |
| **Recall** | how many of the expected records did we find? | ✓ found ÷ expected ids | 1 ÷ 2 = 0.5 |
| **Precision** | how many of the retrieved records are right? | ✓ found ÷ retrieved ids | 1 ÷ 5 = 0.2 |
| **RR** (reciprocal rank) | how **high** was the first right one? | 1 ÷ its place (none = 0) | 1 ÷ 1 = 1.0 |
| **nDCG** | are the right ones near the **top**? | our points ÷ best possible points | 1.0 ÷ 1.63 = 0.61 |

**nDCG, step by step.** A right result in place *p* earns `1 ÷ log2(p + 1)` points: place 1 → 1.0, 2 → 0.63,
3 → 0.5, 4 → 0.43, 5 → 0.39.

```text
our points    = Revenue in place 1                          = 1.0
best possible = both expected records in places 1 and 2      = 1.0 + 0.63 = 1.63
nDCG          = 1.0 ÷ 1.63                                   = 0.61
```

**Why more than one number?** Here RR is perfect (the first result was right) but Recall is only 0.5 — one record
is missing. Each metric sees something different.

**One thing to know:** most questions in our answer key expect only **one** record. For those, Recall is always the
same as Hit, and Precision can never be higher than 1 ÷ 3 (one right out of three). So for one-record questions,
read **Hit** and **RR**; for golden questions (several records), read all five.

The cell below is the one function that calculates all five metrics for one question.

In [13]:
import math


def calculate_metrics(retrieved_ids, expected_ids):
    """
    Calculate five retrieval metrics for one question.

    Args:
        retrieved_ids: IDs returned by search, ordered from best-ranked to
            lowest-ranked. Each position is a rank: the first item is rank 1.
        expected_ids: IDs the answer key considers correct.

    Returns:
        A dictionary containing:
            hit       1 if at least one expected ID was retrieved, else 0
            recall    fraction of expected IDs that were retrieved
            precision fraction of retrieved IDs that were expected
            rr        reciprocal rank of the first expected ID
            ndcg      discounted gain of the result order, normalized against
                      the best possible order

    Assumptions:
        Both lists are non-empty, and each ID appears at most once in each
        list. These assumptions avoid division by zero and keep the counts
        within the usual metric definitions.
    """

    # For every retrieved item, record whether it is in the answer key.
    # The order of these True/False values matches the ranking in
    # retrieved_ids. For example, [False, True, False] means the first
    # correct result appears in position 2.
    is_right = []
    for record_id in retrieved_ids:
        is_right.append(record_id in expected_ids)

    # Count how many retrieved items are considered correct.
    # This count is used by hit, recall, and precision.
    found = is_right.count(True)

    # Hit answers a yes/no question: did search return any correct item?
    # The metric is 1 when at least one correct item was found, otherwise 0.
    if found > 0:
        hit = 1
    else:
        hit = 0

    # Recall measures how much of the answer key was found:
    #     correct retrieved items / all expected items
    # A higher recall means fewer expected items were missed.
    # The question is how much of what you needed came back.
    recall = found / len(expected_ids)

    # Precision measures how accurate the retrieved list was:
    #     correct retrieved items / all retrieved items
    # A higher precision means a larger share of search results were correct.
    # The question is how much of what came back was right.
    precision = found / len(retrieved_ids)

    # Reciprocal rank (RR) rewards finding a correct result near the top.
    # If the first correct result is at position p, RR is 1 / p:
    # position 1 -> 1.0, position 2 -> 0.5, position 3 -> about 0.33.
    # If there are no correct results, RR stays 0.
    rr = 0
    place = 1
    for right in is_right:
        if right:
            rr = 1 / place
            break  # Only the first correct result determines RR.
        place += 1

    # DCG (discounted cumulative gain) gives each correct result a score
    # based on its position. A correct result at position p earns:
    #     1 / log2(p + 1)
    # This gives more credit to results near the top and less credit to
    # correct results farther down the ranking.
    our_points = 0
    place = 1
    for right in is_right:
        if right:
            our_points += 1 / math.log2(place + 1)
        place += 1

    # Calculate the ideal DCG: the score we would get if every available
    # expected result appeared first, in the best possible order.
    #
    # The ideal list cannot contain more results than either:
    #   1. the number of expected IDs, or
    #   2. the number of retrieved positions.
    #
    # Its correct results occupy positions 1, 2, 3, and so on.
    best_points = 0
    for place in range(1, min(len(expected_ids), len(retrieved_ids)) + 1):
        best_points += 1 / math.log2(place + 1)

    # nDCG normalizes the observed DCG against the ideal DCG.
    # A perfect ranking has nDCG 1; a worse ranking has a value closer to 0.
    ndcg = our_points / best_points

    # Return all five metrics together so callers can access them by name.
    return {
        "hit": hit,
        "recall": recall,
        "precision": precision,
        "rr": rr,
        "ndcg": ndcg,
    }


# Example: Revenue is correct and ranked first, while the other expected ID
# (total_amount) is missing. The remaining retrieved IDs are not expected.
example_retrieved = ["glossary::Revenue", "a", "b", "c", "d"]
example_expected = [
    "glossary::Revenue",
    "column::ecom.orders.total_amount",
]

# Print each metric to two decimal places for a compact, readable summary.
for name, value in calculate_metrics(
    example_retrieved, example_expected
).items():
    print(f"  {name:10} {value:.2f}")


# Apply the same calculation to the Chapter 1 question.
# This assumes retrieved_ids and expected_ids have already been set to that
# question's retrieved results and answer-key IDs.
print()
print("Chapter 1 question:", calculate_metrics(retrieved_ids, expected_ids))

  hit        1.00
  recall     0.50
  precision  0.20
  rr         1.00
  ndcg       0.61

Chapter 1 question: {'hit': 0, 'recall': 0.0, 'precision': 0.0, 'rr': 0, 'ndcg': 0.0}


## 3 — All questions

Now the same two steps — **search, then calculate_metrics** — for every question. Each question gets **its own row**
with all five metrics. Each sheet is searched where its expected records live:

| Sheet | Searched in | Retrieved |
|---|---|---|
| glossary | glossary + capability_card | top 3 |
| examples | nlq_examples | top 3 |
| joins | nlq_table + nlq_column + nlq_join | top 3 |
| golden | **the whole knowledge base** (needs records from several tabs) | top 5 |

This makes about 190 searches — about two minutes.

In [14]:
# Each tuple describes one question sheet and how to evaluate its questions:
#   sheet name       label to store in the results
#   questions        rows of questions from that sheet
#   tabs             Pinecone tabs to search, or None to search all tabs
#   expected column  row field containing the answer key IDs
#   top_k            maximum number of results to retrieve per question
SHEETS = [
    ("glossary", glossary_questions, ["glossary", "capability_card"], "right_id", K_ONE),
    ("examples", example_questions, ["nlq_examples"], "right_id", K_ONE),
    ("joins", join_questions, ["nlq_table", "nlq_column", "nlq_join"], "right_id", K_ONE),
    ("golden", golden_questions, None, "must_find", K_MANY),
]

# Collect one result dictionary per evaluated question.
results = []

# Process each sheet using the search settings specified above.
for sheet_name, questions, tabs, expected_column, top_k in SHEETS:

    # Evaluate every question row from the current sheet.
    for row in questions:

        # Read the expected record IDs from this row's configured answer-key
        # column. split_ids converts the stored value into a list of IDs.
        expected_ids = split_ids(row[expected_column])

        # Some golden questions intentionally have no expected records because
        # they test whether the system correctly declines to answer.
        # Skip those here because the retrieval metrics require expected IDs.
        if len(expected_ids) == 0:
            continue

        # STEP 1 — Retrieve the top_k nearest records for this question.
        # The search function applies the configured tab filter, if any, and
        # returns record IDs in ranked order.
        retrieved_ids = search(row["question"], tabs, top_k)

        # STEP 2 — Compare the retrieved IDs with the answer key and calculate
        # hit, recall, precision, reciprocal rank, and nDCG.
        metrics = calculate_metrics(retrieved_ids, expected_ids)

        # STEP 3 — Build a readable result row for this question.
        # Store the IDs as strings separated by " | " so lists are easy to
        # inspect in a table or export to a CSV file.
        result = {
            "sheet": sheet_name,
            "qid": row["qid"],
            "question": row["question"],
            "expected_ids": " | ".join(expected_ids),
            "retrieved_ids": " | ".join(retrieved_ids),
        }

        # Add the five metric values to the same result row.
        result.update(metrics)

        # Keep the completed row for the final summary or export.
        results.append(result)

    # Report when all eligible questions in this sheet have been processed.
    print(f"✅ {sheet_name} done")


# Report the number of question rows evaluated across all sheets.
print(f"\n{len(results)} questions checked — one row each")

✅ glossary done
✅ examples done
✅ joins done
✅ golden done

187 questions checked — one row each


In [15]:
# Inspect the first five question results.
# Each result contains the question, its retrieval metrics, and the IDs used
# to calculate those metrics.
for result in results[:5]:

    # Print the question ID and its five metric values.
    # The :.2f format displays each fractional metric to two decimal places.
    # Include the question text at the end so the scores are easy to interpret.
    print(
        f"{result['qid']}  "
        f"hit {result['hit']}  "
        f"recall {result['recall']:.2f}  "
        f"precision {result['precision']:.2f}"
        f"  rr {result['rr']:.2f}  "
        f"ndcg {result['ndcg']:.2f}   "
        f"{result['question']}"
    )

T01  hit 1  recall 1.00  precision 0.33  rr 1.00  ndcg 1.00   How much did we sell in merchandise, before delivery charges and tax?
T02  hit 1  recall 1.00  precision 0.33  rr 1.00  ndcg 1.00   How much money did we keep once payments that were given back are taken out?
T03  hit 0  recall 0.00  precision 0.00  rr 0.00  ndcg 0.00   How much does a typical checkout bring in?
T04  hit 0  recall 0.00  precision 0.00  rr 0.00  ndcg 0.00   How many pieces of each product went out the door?
T05  hit 1  recall 1.00  precision 0.33  rr 1.00  ndcg 1.00   How much do we earn on a product after what it costs us?


## 4 — Averages

For each sheet, each metric is averaged over its question rows. The average of **RR** has its own name: **MRR**
(mean reciprocal rank). **1.0 = perfect.**

In [16]:
# These are the metrics to summarize for each sheet.
# "rr" is displayed as MRR (mean reciprocal rank) because this value is
# averaged across multiple questions.
METRICS = ["hit", "recall", "precision", "rr", "ndcg"]

# Store each sheet's average metric values here.
# The structure will look like:
# {
#     "glossary": {"hit": 0.8, "recall": 0.6, ...},
#     "examples": {"hit": 0.9, "recall": 0.7, ...},
# }
AVERAGES = {}

# Build and print the table header.
# "sheet" is left-aligned in a 10-character-wide field.
# "questions" is right-aligned in a 10-character-wide field.
header = f"{'sheet':10}{'questions':>10}"

# Add one right-aligned column heading for each metric.
# Label the average of reciprocal rank as MRR in the output table.
for metric in METRICS:
    if metric == "rr":
        header += f"{'MRR':>11}"
    else:
        header += f"{metric:>11}"

print(header)


# Print one summary line for each sheet listed in SHEETS.
for sheet_name, questions, tabs, expected_column, top_k in SHEETS:

    # STEP 1 — Collect the evaluated result rows for this sheet.
    # The original results list contains questions from all sheets, so keep
    # only rows whose "sheet" value matches the current sheet name.
    rows = []
    for result in results:
        if result["sheet"] == sheet_name:
            rows.append(result)

    # STEP 2 — Calculate the average of each metric for this sheet.
    # Start a dictionary for this sheet, then sum each metric across its rows
    # and divide by the number of rows to get the mean.
    AVERAGES[sheet_name] = {}
    for metric in METRICS:
        total = 0
        for row in rows:
            total += row[metric]

        # Assumes this sheet has at least one evaluated row; otherwise,
        # dividing by len(rows) would raise a ZeroDivisionError.
        AVERAGES[sheet_name][metric] = total / len(rows)

    # STEP 3 — Format and print this sheet's summary line.
    # The question count is the number of evaluated rows in this sheet.
    line = f"{sheet_name:10}{len(rows):>10}"

    # Add each average using two digits after the decimal point.
    for metric in METRICS:
        line += f"{AVERAGES[sheet_name][metric]:>11.2f}"

    print(line)

sheet      questions        hit     recall  precision        MRR       ndcg
glossary          66       0.94       0.94       0.31       0.81       0.84
examples          61       1.00       1.00       0.33       0.99       0.99
joins             16       0.31       0.31       0.10       0.17       0.20
golden            44       0.68       0.38       0.18       0.32       0.28


**How to read it:**

- **glossary · examples · joins** expect one record per question → read **hit** (how often it was in the top 3) and
  **MRR** (how high). Their recall equals their hit, and their precision can't go above 0.33.
- **golden** expects several records → read all five. **recall** says how much of what a business question needs
  came back from one search.
- **examples** usually scores highest (example questions sound like users); **joins** usually lowest (all join
  texts look alike: "Join A to B on …").
- Two golden questions can never reach recall 1.0: G26 expects 6 records and G28 expects 7, but we retrieve 5.

Now the most useful part: **which questions failed?**

In [17]:
# Find questions where retrieval returned no expected records.
# A hit value of 0 means none of the retrieved IDs matched the answer key.
failed = []
for result in results:
    if result["hit"] == 0:
        failed.append(result)

# Report the total number of questions with no correct retrieval.
# Then show up to the first 10 so the misses can be inspected.
print(f"{len(failed)} questions found nothing right. The first 10:\n")

for result in failed[:10]:
    # Show which sheet and question produced the miss.
    print(f"{result['sheet']:9} {result['qid']}  {result['question']}")

    # Show the answer-key IDs that search was expected to find.
    print(f"           expected:  {result['expected_ids']}")

    # Show the IDs search actually returned, in ranked order.
    print(f"           retrieved: {result['retrieved_ids']}")

    # Add a blank line to separate this question from the next one.
    print()

29 questions found nothing right. The first 10:

glossary  T03  How much does a typical checkout bring in?
           expected:  glossary::Average Order Value
           retrieved: glossary::Revenue | capability_card::3 | glossary::Percentage Coupon

glossary  T04  How many pieces of each product went out the door?
           expected:  glossary::Units Sold
           retrieved: capability_card::1 | capability_card::3 | glossary::Revenue

glossary  T32  Which suppliers expect payment within two weeks?
           expected:  glossary::Net 15
           retrieved: glossary::2/10 Net 30 | glossary::Net 60 | glossary::Net 30

glossary  T33  Which suppliers give us a month and a half to pay?
           expected:  glossary::Net 45
           retrieved: glossary::Net 30 | glossary::Net 60 | glossary::2/10 Net 30

joins     J02  For each product line, on what date was its order placed?
           expected:  join::ecom.order_items->ecom.orders::order_id
           retrieved: column::ecom.product

In [18]:
# Ensure the output folder exists.
# exist_ok=True means Python will reuse it if it has already been created.
os.makedirs("eval_runs", exist_ok=True)

# Create a timestamped filename so each evaluation run gets its own CSV.
# The timestamp format is run_YYYYMMDD_HHMMSS.
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
results_file = f"eval_runs/{run_name}.csv"

# Open the output file for writing.
# newline="" prevents extra blank lines between CSV rows on some systems.
# utf-8 supports question text and IDs containing non-ASCII characters.
with open(results_file, "w", newline="", encoding="utf-8") as file:

    # Use the keys from the first result row as the CSV column headings.
    # This assumes results contains at least one row.
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()))

    # Write the column headings as the first row of the CSV.
    writer.writeheader()

    # Write one CSV row for each evaluated question.
    writer.writerows(results)

# Confirm where the file was saved and how many question rows it contains.
print("✅ Saved", results_file, "—", len(results), "rows, one per question")

✅ Saved eval_runs/run_20261122_102644.csv — 187 rows, one per question


**When a question fails**, open its record in `kb_feed_ecom.xlsx` and compare it with the question:

| What you see | Fix |
|---|---|
| the record's text doesn't use the user's words | add them to `synonyms` / `analyst_synonyms` |
| the record's text is only a formula or code | rewrite `meaning` / `description` in plain words |
| the "wrong" result is really a better answer | fix the answer key |

Then reload in notebook 02 and run this notebook again. ⚠️ Never paste the test question itself into the Excel —
it would pass by copying, and prove nothing.

## 5 — Save the run to MLflow

MLflow keeps every run, so you can compare them side by side — for example before and after a fix.
See `README_MLflow.md` for how to open it in your browser.

```text
MLflow run
 ├── parameters   settings      embed_model, k_one, k_many
 ├── metrics      averages      glossary_hit, glossary_mrr, … golden_recall, golden_ndcg  (5 per sheet)
 └── artifact     a file        the CSV with one row per question
```

In [19]:
import mlflow

# Store MLflow tracking data in a local SQLite database named mlflow.db.
# The relative path means the database is created in the notebook's current
# working directory.
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Select the experiment that will group these retrieval evaluation runs.
# MLflow creates the experiment if it does not already exist.
mlflow.set_experiment("vectordb-03-retrieval-eval")

# Start a new run and use the same timestamped name as the CSV file.
# All parameters, metrics, and artifacts logged inside this block belong
# to this run.
with mlflow.start_run(run_name=run_name):

    # STEP 1 — Log the configuration parameters used for this evaluation.
    # Parameters describe the run settings, such as the embedding model and
    # the number of records retrieved for each question type.
    mlflow.log_param("embed_model", EMBED_MODEL)
    mlflow.log_param("k_one", K_ONE)
    mlflow.log_param("k_many", K_MANY)

    # STEP 2 — Log the average metrics for each sheet.
    # Each metric name includes the sheet name, for example "glossary_hit"
    # or "golden_recall", so values from different sheets stay distinct.
    # The average reciprocal rank ("rr") is named "mrr" in MLflow because it
    # is the mean reciprocal rank across the questions in that sheet.
    for sheet_name in AVERAGES:
        for metric in AVERAGES[sheet_name]:
            value = AVERAGES[sheet_name][metric]

            if metric == "rr":
                mlflow.log_metric(sheet_name + "_mrr", value)
            else:
                mlflow.log_metric(sheet_name + "_" + metric, value)

    # STEP 3 — Attach the per-question CSV to this run as an artifact.
    # The CSV contains each question, its expected and retrieved IDs, and its
    # individual metric values.
    mlflow.log_artifact(results_file)

# Confirm that the run was recorded under the chosen name.
print("✅ Saved to MLflow as", run_name)

# To open the MLflow tracking UI in a terminal, run:
# mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
# Then follow the instructions in README_MLflow.md to view the experiment.

2026/11/22 10:26:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/22 10:26:46 INFO mlflow.store.db.utils: Updating database tables
2026/11/22 10:26:46 INFO mlflow.tracking.fluent: Experiment with name 'vectordb-03-retrieval-eval' does not exist. Creating a new experiment.


✅ Saved to MLflow as run_20261122_102644


## What you learned

An **answer key** (questions + right ids) → the **same search** the agent uses → **compare** the ids → metrics for
**each question** → **averages** per sheet → the **failed questions** tell you what to fix → **MLflow** keeps every run.

One function, five standard retrieval metrics — **hit, recall, precision, RR (→ MRR), nDCG** — the same ones used in
any RAG project.

## Quick check

1. A glossary question's expected record is in 2nd place. What are hit, recall, precision and rr?
2. Why is recall always the same as hit for glossary questions?
3. A golden question has rr = 1.0 but recall = 0.5. What happened?
4. What is MRR?
5. A question fails. Do you fix the code or the Excel file?

<details><summary><b>Answers</b> (click)</summary>

1. hit = 1, recall = 1 ÷ 1 = 1.0, precision = 1 ÷ 3 = 0.33, rr = 1 ÷ 2 = 0.5.
2. There is only one expected record: found it → recall 1, didn't → recall 0. Exactly like hit.
3. The first result was right, but half of the expected records are missing.
4. The average of rr over all questions of a sheet.
5. Usually the Excel file: a missing word in the synonyms, or a meaning written only as a formula.

</details>

**Next:** Agent Version 3 uses this search as a tool — and then we also judge the **answers** (with an LLM judge).